# mPES - Colab Launcher desde GitHub

Este notebook clona una copia ligera del repositorio en el almacenamiento local
de Colab y ejecuta la optimizacion desde `h1/`. Los resultados se conservan
en Google Drive.

Configura en la primera celda el repositorio y la rama. El clonado usa
`--depth 1 --single-branch` para reducir tiempo, espacio y trafico de red.
Para `ens_sprb` o `ens_accq`, los modelos pre-entrenados viajan dentro del
propio clon (`h1/ml/pes_{dqn,rdqn,trf}/inputs/*_model.keras`); si en Drive hay
copias mas recientes (`MyDrive/mPES/<pes_dqn|pes_rdqn|pes_trf>/`, donde las
guarda `retrain_gpu.ipynb`), esas tienen prioridad y se copian al clon. Los
modelos nunca se reentrenan desde este notebook.

Ejecuta todas las celdas en orden. Para ejecuciones largas, activa Background
execution en la sesion de Colab.

In [ ]:
"""Colab launcher for mPES Bayesian optimisation."""
# Mount Google Drive before cloning the repository.
# ==========================================================================
# MOUNT GOOGLE DRIVE
# ==========================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

In [ ]:
REPOSITORY   = 'https://github.com/Maximiliano0/mPES_2026.git'
BRANCH       = 'new_uq'
WORKSPACE    = '/content/mPES'
H_DIR        = os.path.join(WORKSPACE, 'h1')
UTILS_DIR    = os.path.join(WORKSPACE, 'utils')
OUTPUT_ROOT  = '/content/drive/MyDrive/mPES/runs'
PKG          = 'ens_sprb'  # ql | dql | dqn | rdqn | ac | tr | ens_sprb | ens_accq
N_TRIALS     = 50
RESUME_DATE  = ''  # YYYY-MM-DD to resume, or '' for a new run
USE_GPU      = 0

valid_packages = ('ql', 'dql', 'dqn', 'rdqn', 'ac', 'tr', 'ens_sprb', 'ens_accq')
if PKG not in valid_packages:
    raise ValueError(f'Unsupported PKG: {PKG!r}')

os.environ.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
print(f'[INFO] [Celda 1] Configuración guardada: PKG={PKG!r}, N_TRIALS={N_TRIALS}, BRANCH={BRANCH!r}')

In [ ]:
# Shallow clone: only the selected branch and its current snapshot.
import os
import subprocess

if os.path.isdir(WORKSPACE):
    print(f"[INFO] [Celda 2] Borrando workspace anterior en {WORKSPACE}...")
    subprocess.run(['rm', '-rf', WORKSPACE], check=True)

print(f"[INFO] [Celda 2] Clonando el repositorio rama '{BRANCH}'...")
subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
    REPOSITORY, WORKSPACE,
], check=True)

print("[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...")
setup = subprocess.run(
    ['bash', os.path.join(H_DIR, 'general', 'colab', 'setup_colab.sh')],
    check=False,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(setup.stdout)
if setup.returncode != 0:
    raise RuntimeError(f'setup_colab.sh exited with code {setup.returncode}')
print(f'[INFO] [Celda 2] Shallow clone y setup completados: {REPOSITORY}@{BRANCH}')

In [ ]:
# Resolve pretrained models for ensemble packages — never retrain them here.
# The canonical models ship inside the clone at h1/ml/<pkg>/inputs/. If Drive
# holds newer copies (e.g. from retrain_gpu.ipynb under MyDrive/mPES/<pkg>/),
# they take precedence and are copied over the clone's baseline.
import glob
import os
import shutil

DRIVE_ROOT = '/content/drive/MyDrive/mPES'
ENSEMBLE_MODELS = {'dqn': 'pes_dqn', 'rdqn': 'pes_rdqn', 'trf': 'pes_trf'}

In [ ]:
import os
import subprocess

run_environment = os.environ.copy()